# HF Korean Project v3: Qwen3.5-4B paraphrase 생성 + STS reranking

이 노트북은 KLUE-STS의 높은 유사도 문장쌍을 paraphrase 생성 데이터로 바꾸고, `Qwen3.5-4B` 계열 모델을 LoRA/QLoRA 방식으로 SFT한 뒤, 생성 후보 30개를 기존 STS 회귀 모델로 채점해 top 10을 보여주는 실험입니다.

필요 라이브러리

uv pip install trl peft bitsandbytes 

## 라이브러리 의존성 및 모델 선택 메모

이 노트북은 `transformers`, `datasets`, `accelerate`, `torch` 외에 QLoRA/SFT용 `trl`, `peft`, `bitsandbytes`가 필요합니다. 설치가 필요하다면 코드 셀 대신 터미널에서 `uv pip install trl peft bitsandbytes`를 실행하세요.

`Qwen/Qwen3.5-4B`는 공식 모델 카드에서 image-text-to-text 사용 예시를 제공하지만, config상 causal LM으로도 로드할 수 있습니다. fine-tuning 목적에는 공식 카드가 “fine-tuning, in-context learning experiments” 용도라고 설명하는 `Qwen/Qwen3.5-4B-Base`를 기본값으로 사용합니다. post-trained 모델로 바꾸고 싶다면 `QWEN_MODEL_NAME`만 `Qwen/Qwen3.5-4B`로 바꿔 실험하면 됩니다.

# 0. 실행 환경 확인

QLoRA SFT에 필요한 패키지가 있는지 먼저 확인합니다. 패키지가 없으면 이후 학습 셀에서 긴 traceback이 나기 전에 필요한 설치 항목을 명확히 알려줍니다.

In [1]:
import importlib.util
from pathlib import Path

required_packages = ["trl", "peft", "bitsandbytes"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing_packages:
    raise ModuleNotFoundError(
        "다음 패키지가 필요합니다: "
        + ", ".join(missing_packages)
        + "\n터미널에서 `uv pip install trl peft bitsandbytes`를 실행한 뒤 커널을 재시작하세요."
    )

print("QLoRA/SFT 필수 패키지 확인 완료")

QLoRA/SFT 필수 패키지 확인 완료


# 1. KLUE-STS 로드

Hugging Face `datasets`에서 KLUE-STS를 불러옵니다. 이번 실험에서는 원래의 이진 분류 라벨보다 `real-label` 점수를 사용해 paraphrase 학습 데이터를 고릅니다.

In [2]:
from datasets import Dataset, DatasetDict, load_dataset

klue_sts_dataset = load_dataset("klue", "sts")
print(klue_sts_dataset)
print(klue_sts_dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['guid', 'source', 'sentence1', 'sentence2', 'labels'],
        num_rows: 11668
    })
    validation: Dataset({
        features: ['guid', 'source', 'sentence1', 'sentence2', 'labels'],
        num_rows: 519
    })
})
{'guid': 'klue-sts-v1_train_00000', 'source': 'airbnb-rtt', 'sentence1': '숙소 위치는 찾기 쉽고 일반적인 한국의 반지하 숙소입니다.', 'sentence2': '숙박시설의 위치는 쉽게 찾을 수 있고 한국의 대표적인 반지하 숙박시설입니다.', 'labels': {'label': 3.7, 'real-label': 3.714285714285714, 'binary-label': 1}}


# 2. real-label >= 4.0 문장쌍을 paraphrase pair로 변환

`real-label`이 4.0 이상인 문장쌍만 의미가 거의 같은 paraphrase 쌍으로 봅니다. 각 샘플은 `문장1 -> 문장2`, `문장2 -> 문장1` 양방향으로 만들어 데이터 수를 늘립니다. 애매하거나 유사하지 않은 문장쌍은 생성 학습에 넣지 않습니다.

In [3]:
REAL_LABEL_MIN = 4.0


def build_paraphrase_prompt(sentence):
    return (
        "### 지시\n"
        "다음 한국어 문장을 의미는 유지하면서 다른 표현의 한 문장으로 바꿔 쓰세요.\n"
        "출력은 paraphrase 문장 하나만 작성하세요.\n\n"
        "### 입력\n"
        f"{sentence}\n\n"
        "### 출력\n"
    )


def make_paraphrase_records(split_dataset, real_label_min=REAL_LABEL_MIN):
    records = []
    seen = set()

    for example in split_dataset:
        score = float(example["labels"]["real-label"])
        if score < real_label_min:
            continue

        sentence1 = example["sentence1"].strip()
        sentence2 = example["sentence2"].strip()
        if not sentence1 or not sentence2 or sentence1 == sentence2:
            continue

        for source, target in [(sentence1, sentence2), (sentence2, sentence1)]:
            key = (source, target)
            if key in seen:
                continue
            seen.add(key)
            records.append(
                {
                    "source": source,
                    "target": target,
                    "score": score,
                    "prompt": build_paraphrase_prompt(source),
                    "completion": target,
                }
            )

    return records


train_records = make_paraphrase_records(klue_sts_dataset["train"])
test_records = make_paraphrase_records(klue_sts_dataset["validation"])

paraphrase_dataset = DatasetDict(
    {
        "train": Dataset.from_list(train_records),
        "test": Dataset.from_list(test_records),
    }
)

print(paraphrase_dataset)
print(paraphrase_dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['source', 'target', 'score', 'prompt', 'completion'],
        num_rows: 5456
    })
    test: Dataset({
        features: ['source', 'target', 'score', 'prompt', 'completion'],
        num_rows: 220
    })
})
{'source': '호스트의 답장이 늦으나, 개선될 것으로 보입니다.', 'target': '호스트 응답이 늦었지만 개선될 것으로 보입니다.', 'score': 4.714285714285714, 'prompt': '### 지시\n다음 한국어 문장을 의미는 유지하면서 다른 표현의 한 문장으로 바꿔 쓰세요.\n출력은 paraphrase 문장 하나만 작성하세요.\n\n### 입력\n호스트의 답장이 늦으나, 개선될 것으로 보입니다.\n\n### 출력\n', 'completion': '호스트 응답이 늦었지만 개선될 것으로 보입니다.'}


학습 시간을 조절할 수 있도록 최대 학습 샘플 수 변수를 둡니다. 처음 디버깅할 때는 `MAX_TRAIN_SAMPLES = 512`처럼 줄여 실행하고, 정상 동작을 확인한 뒤 `None`으로 전체 학습을 진행하면 됩니다.

In [4]:
MAX_TRAIN_SAMPLES = None

if MAX_TRAIN_SAMPLES is not None:
    paraphrase_train_dataset = paraphrase_dataset["train"].shuffle(seed=42).select(range(MAX_TRAIN_SAMPLES))
else:
    paraphrase_train_dataset = paraphrase_dataset["train"].shuffle(seed=42)

paraphrase_eval_dataset = paraphrase_dataset["test"]

print("train rows:", len(paraphrase_train_dataset))
print("eval rows:", len(paraphrase_eval_dataset))

train rows: 5456
eval rows: 220


# 3. Qwen3.5-4B LoRA/QLoRA SFT 준비

Qwen3.5-4B는 크기가 크므로 full fine-tuning 대신 4-bit QLoRA와 LoRA adapter를 사용합니다. TRL의 `SFTTrainer`는 prompt-completion 데이터셋을 지원하고, PEFT 설정을 넘기면 adapter만 학습할 수 있습니다.

In [5]:
import gc
import math
import os
import random
import re
import time

import numpy as np
import torch
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

QWEN_MODEL_NAME = "Qwen/Qwen3.5-4B-Base"
QWEN_OUTPUT_DIR = "qwen3_5_4b_klue_sts_paraphrase_sft"
QWEN_ADAPTER_DIR = "qwen3_5_4b_klue_sts_paraphrase_adapter"

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

print(f"bf16 사용여부: {use_bf16} / fp16 사용여부: {use_fp16}")
print("compute dtype:", compute_dtype)

bf16 사용여부: True / fp16 사용여부: False
compute dtype: torch.bfloat16


Qwen 토크나이저와 4-bit quantization 설정을 준비합니다. causal LM 계열은 padding token 설정이 중요하므로, tokenizer와 model config 모두에 `pad_token_id`를 맞춥니다.

In [6]:
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, trust_remote_code=True)

if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("pad_token:", qwen_tokenizer.pad_token, qwen_tokenizer.pad_token_id)
print("eos_token:", qwen_tokenizer.eos_token, qwen_tokenizer.eos_token_id)

pad_token: <|endoftext|> 248044
eos_token: <|endoftext|> 248044


학습용 completion에는 EOS 토큰을 붙여 생성이 적절히 끝나도록 합니다. prompt-completion 형식을 쓰면 SFTTrainer가 completion 부분 위주로 loss를 계산할 수 있습니다.

In [7]:
def add_eos_to_completion(example):
    completion = example["completion"].strip()
    if not completion.endswith(qwen_tokenizer.eos_token):
        completion = completion + qwen_tokenizer.eos_token
    return {"completion": completion}


paraphrase_train_dataset = paraphrase_train_dataset.map(add_eos_to_completion)
paraphrase_eval_dataset = paraphrase_eval_dataset.map(add_eos_to_completion)

print(paraphrase_train_dataset[0]["prompt"])
print(paraphrase_train_dataset[0]["completion"])

Map:   0%|          | 0/5456 [00:00<?, ? examples/s]

Map:   0%|          | 0/220 [00:00<?, ? examples/s]

### 지시
다음 한국어 문장을 의미는 유지하면서 다른 표현의 한 문장으로 바꿔 쓰세요.
출력은 paraphrase 문장 하나만 작성하세요.

### 입력
너가 생각했을때 넌 광고메일이 더 짜증나는 거 같아, 결제 메일이 더 짜증나는거 같아?

### 출력

당신은 광고랑 결제 메일 중에 더 짜증나는게 뭐에요?<|endoftext|>


4-bit로 base model을 로드하고 k-bit training 준비를 적용합니다. `use_cache=False`와 gradient checkpointing은 학습 메모리 사용량을 줄이기 위해 필요합니다.

In [8]:
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=compute_dtype,
    trust_remote_code=True,
)

qwen_model.config.use_cache = False
qwen_model.config.pad_token_id = qwen_tokenizer.pad_token_id
qwen_model = prepare_model_for_kbit_training(qwen_model)

print(type(qwen_model))

Exception in thread Thread-5 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Sungwoo\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "C:\Users\Sungwoo\AppData\Local\Programs\Python\Python310\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Sungwoo\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1499, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\Sungwoo\AppData\Local\Programs\Python\Python310\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte


model.safetensors.index.json: 0.00B [00:00, ?B/s]

f:\code\AIFFEL\AIFFEL_study\.venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sungwoo\.cache\huggingface\hub\models--Qwen--Qwen3.5-4B-Base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

<class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5ForCausalLM'>


LoRA 설정입니다. Qwen3.5의 세부 모듈명이 바뀌어도 덜 깨지도록 PEFT가 지원하는 `target_modules='all-linear'`를 사용합니다. adapter 학습이므로 learning rate는 일반 full fine-tuning보다 높은 `1e-4`를 기본값으로 둡니다.

In [9]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

sft_args = SFTConfig(
    output_dir=QWEN_OUTPUT_DIR,
    max_length=512,
    dataset_text_field="text",
    completion_only_loss=True,
    packing=False,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    bf16=use_bf16,
    fp16=use_fp16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    run_name="qwen3.5-4b-klue-sts-paraphrase",
    seed=42,
)

trainer = SFTTrainer(
    model=qwen_model,
    args=sft_args,
    train_dataset=paraphrase_train_dataset,
    eval_dataset=paraphrase_eval_dataset,
    processing_class=qwen_tokenizer,
    peft_config=lora_config,
)

print("SFTTrainer 준비 완료")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/5456 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5456 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

SFTTrainer 준비 완료


SFT 학습을 실행하고 LoRA adapter를 저장합니다. 4B 모델 학습은 시간이 걸리므로, 처음에는 `MAX_TRAIN_SAMPLES`를 작게 둔 상태에서 이 셀을 테스트하는 것을 권장합니다.

In [10]:
start_time = time.time()
train_result = trainer.train()
train_seconds = time.time() - start_time

trainer.save_model(QWEN_ADAPTER_DIR)
qwen_tokenizer.save_pretrained(QWEN_ADAPTER_DIR)

print(f"SFT 학습 시간: {train_seconds:.2f}초")
print(train_result.metrics)
print("adapter 저장 경로:", QWEN_ADAPTER_DIR)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 248044}.


Epoch,Training Loss,Validation Loss
1,1.013295,1.159228
2,0.723310,1.156065


SFT 학습 시간: 20314.66초
{'train_runtime': 20314.4452, 'train_samples_per_second': 0.537, 'train_steps_per_second': 0.067, 'total_flos': 1.856598921024307e+16, 'train_loss': 0.9043626500364622}
adapter 저장 경로: qwen3_5_4b_klue_sts_paraphrase_adapter


# 4. 입력 문장으로 paraphrase 후보 30개 생성

학습된 LoRA adapter를 사용해 입력 문장 하나에서 paraphrase 후보를 여러 개 생성합니다. 한 번에 30개를 만들면 VRAM 사용량이 커질 수 있어 내부적으로 여러 batch로 나누어 생성합니다.

In [11]:
def load_qwen_paraphrase_model(adapter_dir=QWEN_ADAPTER_DIR):
    base_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=compute_dtype,
        trust_remote_code=True,
    )
    base_model.config.use_cache = True
    base_model.config.pad_token_id = qwen_tokenizer.pad_token_id
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()
    return model


def clean_paraphrase_output(text):
    text = text.strip()
    text = re.split(r"\n###|###|\n입력:|\n지시:", text)[0].strip()
    text = text.replace(qwen_tokenizer.eos_token or "", "").strip()
    lines = [line.strip(" -•\t") for line in text.splitlines() if line.strip()]
    return lines[0] if lines else text


def generate_paraphrase_candidates(
    sentence,
    model=None,
    num_candidates=30,
    batch_return_sequences=5,
    max_new_tokens=80,
    temperature=0.85,
    top_p=0.92,
    repetition_penalty=1.05,
):
    if model is None:
        model = load_qwen_paraphrase_model()

    prompt = build_paraphrase_prompt(sentence)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(model.device)
    candidates = []
    seen = set()

    while len(candidates) < num_candidates:
        remaining = num_candidates - len(candidates)
        current_batch = min(batch_return_sequences, remaining)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                repetition_penalty=repetition_penalty,
                max_new_tokens=max_new_tokens,
                num_return_sequences=current_batch,
                pad_token_id=qwen_tokenizer.pad_token_id,
                eos_token_id=qwen_tokenizer.eos_token_id,
            )

        input_length = inputs["input_ids"].shape[-1]
        for output in outputs:
            generated = qwen_tokenizer.decode(output[input_length:], skip_special_tokens=True)
            candidate = clean_paraphrase_output(generated)
            if candidate and candidate != sentence and candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

        if len(seen) == 0 and len(candidates) == 0:
            break

    return candidates[:num_candidates]

# 5. 기존 STS regression 모델로 후보 점수 계산

v2에서 학습한 `klue/roberta-base` 회귀 scorer를 불러옵니다. 이 모델은 `문장1, 후보문장` 쌍의 유사도 점수를 0~5 범위로 예측하고, 생성 후보를 정렬하는 데 사용합니다.

In [12]:
from transformers import AutoModelForSequenceClassification

STS_SCORER_TOKENIZER_NAME = "klue/roberta-base"


def find_sts_scorer_model_dir():
    search_roots = [Path.cwd(), Path.cwd() / "huggingface_transformers_project"]
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        candidates.extend(root.glob("transformers_klue_sts_regression_roberta_lr_*/best_model"))

    existing = [path for path in candidates if (path / "config.json").exists()]
    if not existing:
        raise FileNotFoundError(
            "v2에서 학습한 STS 회귀 모델을 찾지 못했습니다. "
            "`STS_SCORER_MODEL_DIR`에 best_model 경로를 직접 지정하세요."
        )

    return str(max(existing, key=lambda path: path.stat().st_mtime))


STS_SCORER_MODEL_DIR = find_sts_scorer_model_dir()
print("STS scorer 경로:", STS_SCORER_MODEL_DIR)

STS scorer 경로: f:\code\AIFFEL\AIFFEL_study\huggingface_transformers_project\transformers_klue_sts_regression_roberta_lr_5em05\best_model


STS scorer 모델과 토크나이저를 로드합니다. RoBERTa 계열이므로 `token_type_ids`는 만들지 않도록 설정합니다.

In [13]:
sts_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sts_tokenizer = AutoTokenizer.from_pretrained(STS_SCORER_TOKENIZER_NAME)
sts_model = AutoModelForSequenceClassification.from_pretrained(STS_SCORER_MODEL_DIR).to(sts_device)
sts_model.eval()


def score_paraphrase_candidates(source_sentence, candidates, batch_size=32, max_length=128):
    scored = []

    for start in range(0, len(candidates), batch_size):
        batch_candidates = candidates[start : start + batch_size]
        encoded = sts_tokenizer(
            [source_sentence] * len(batch_candidates),
            batch_candidates,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_token_type_ids=False,
            return_tensors="pt",
        )
        encoded = {key: value.to(sts_device) for key, value in encoded.items()}

        with torch.no_grad():
            outputs = sts_model(**encoded)

        scores = outputs.logits.squeeze(-1).detach().cpu().numpy()
        scores = np.clip(scores, 0.0, 5.0)

        for candidate, score in zip(batch_candidates, scores):
            scored.append({"candidate": candidate, "score": float(score)})

    return scored

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

# 6. top 10 정렬 출력

입력 문장 하나를 받아 Qwen paraphrase 후보 30개를 만들고, STS scorer 점수 기준으로 높은 순서의 top 10만 반환하는 최종 함수입니다.

In [14]:
def generate_and_rank_paraphrases(
    sentence,
    qwen_model_for_generation=None,
    num_candidates=30,
    top_k=10,
):
    candidates = generate_paraphrase_candidates(
        sentence=sentence,
        model=qwen_model_for_generation,
        num_candidates=num_candidates,
    )
    scored_candidates = score_paraphrase_candidates(sentence, candidates)
    ranked = sorted(scored_candidates, key=lambda row: row["score"], reverse=True)
    return ranked[:top_k]


def print_ranked_paraphrases(ranked_results):
    for rank, row in enumerate(ranked_results, start=1):
        print(f"{rank:02d}. score={row['score']:.4f} | {row['candidate']}")

아래 셀에서 직접 문장을 바꿔 paraphrase 생성과 reranking을 테스트합니다. 이미 위에서 학습한 adapter가 메모리에 있다면 그대로 쓰고, 커널을 재시작한 뒤라면 `load_qwen_paraphrase_model()`로 adapter를 다시 불러옵니다.

In [15]:
input_sentence = "숙소 위치는 찾기 쉽고 일반적인 한국의 반지하 숙소입니다."

qwen_generation_model = load_qwen_paraphrase_model(QWEN_ADAPTER_DIR)
ranked_results = generate_and_rank_paraphrases(
    sentence=input_sentence,
    qwen_model_for_generation=qwen_generation_model,
    num_candidates=30,
    top_k=10,
)

print("입력 문장:", input_sentence)
print()
print_ranked_paraphrases(ranked_results)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

입력 문장: 숙소 위치는 찾기 쉽고 일반적인 한국의 반지하 숙소입니다.

01. score=4.9215 | 숙소의 위치는 찾기 쉽고 한국의 일반적인 반지하 숙소에요.
02. score=4.9193 | 숙소의 위치는 찾기가 쉽고 한국의 일반적인 반지하 숙소에요.
03. score=4.9134 | 숙소의 위치는 찾기가 쉽고 일반적인 한국 반지하 숙소입니다.
04. score=4.9123 | 숙소의 위치를 찾기가 쉽고, 한국의 일반적인 반지하 숙소에요.
05. score=4.9082 | 숙소 위치 찾기가 쉽고 한국에서 일반적인 반지하 숙소입니다.
06. score=4.8898 | 숙소의 위치를 찾기가 쉽고 한국에서 일반적인 반지하 숙소입니다.
07. score=4.8650 | 숙소의 위치가 쉽게 찾을 수 있고, 일반적인 한국 반지하 숙소입니다.
08. score=4.7484 | 숙소의 위치를 찾아내기 쉽고, 한국 일반적인 반지하 숙박시설입니다.
09. score=4.6906 | 숙박시설의 위치는 쉽게 찾을 수 있으며, 한국의 일반적인 반지하 숙박 시설입니다.
10. score=4.6371 | 숙소의 위치는 찾기 쉬워요. 한국 일반적인 반지하 숙박시설이에요.


실험이 끝난 뒤 GPU 메모리를 정리합니다. 다음 실험을 이어서 할 때 VRAM이 남아 생기는 문제를 줄이기 위한 셀입니다.

In [16]:
try:
    del qwen_generation_model
except NameError:
    pass

try:
    del trainer
    del qwen_model
except NameError:
    pass

try:
    del sts_model
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("메모리 정리 완료")

메모리 정리 완료


In [17]:
input_sentence = "박대건님의 달콤 살벌한 스토리르 듣기위해서는 제주도를 와야만 한다."

qwen_generation_model = load_qwen_paraphrase_model(QWEN_ADAPTER_DIR)
ranked_results = generate_and_rank_paraphrases(
    sentence=input_sentence,
    qwen_model_for_generation=qwen_generation_model,
    num_candidates=30,
    top_k=30,
)

print("입력 문장:", input_sentence)
print()
print_ranked_paraphrases(ranked_results)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

NameError: name 'sts_model' is not defined

# 7. 저장된 LoRA adapter와 STS scorer 다시 불러오기

LoRA/QLoRA 학습에서 `trainer.save_model(QWEN_ADAPTER_DIR)`로 저장되는 것은 보통 전체 Qwen 4B 모델이 아니라 **LoRA adapter 가중치와 adapter 설정**입니다. 따라서 다음에 사용할 때는 base model(`Qwen/Qwen3.5-4B-Base`)을 다시 로드한 뒤, `PeftModel.from_pretrained(base_model, QWEN_ADAPTER_DIR)`로 adapter를 얹어야 합니다. base model까지 하나로 합친 merged model을 따로 저장하지 않았다면 adapter 폴더만으로는 단독 추론할 수 없습니다.

아래 셀은 마지막 메모리 정리 셀 때문에 삭제된 `sts_model`도 다시 불러오고, 저장된 Qwen LoRA adapter도 다시 불러와서 예문 생성과 top 10 정렬을 바로 실행할 수 있게 해줍니다.

In [ ]:
QWEN_MODEL_NAME = "Qwen/Qwen3.5-4B-Base"
QWEN_OUTPUT_DIR = "qwen3_5_4b_klue_sts_paraphrase_sft"
QWEN_ADAPTER_DIR = "qwen3_5_4b_klue_sts_paraphrase_adapter"

STS_SCORER_TOKENIZER_NAME = "klue/roberta-base"
STS_SCORER_MODEL_DIR = "f:\code\AIFFEL\AIFFEL_study\huggingface_transformers_project\transformers_klue_sts_regression_roberta_lr_5em05\best_model"

def reload_saved_qwen_adapter(adapter_dir=QWEN_ADAPTER_DIR):
    """저장된 LoRA adapter를 base Qwen 모델 위에 다시 로드합니다."""
    from peft import PeftModel
    from transformers import AutoModelForCausalLM

    base_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=compute_dtype,
        trust_remote_code=True,
    )
    base_model.config.use_cache = True      # 추론용이므로 KV 캐시를 켭니다. (학습 때는 보통 끕니다.)
    
    # 패딩 토큰 ID를 config에 명시합니다. 배치 생성 시 경고를 줄이는 역할입니다. 여기서 전역 qwen_tokenizer에 의존합니다.
    base_model.config.pad_token_id = qwen_tokenizer.pad_token_id

    # base 모델 위에 LoRA adapter를 입혀 PeftModel을 만들고, eval()로 평가 모드(dropout 비활성 등) 전환 후 반환합니다.
    # merge_and_unload()로 병합하는 선택지도 있지만, 양자화된 base 위에서는 병합이 제약될 수 있습니다.
    loaded_model = PeftModel.from_pretrained(base_model, adapter_dir)
    loaded_model.eval()
    return loaded_model


def reload_sts_scorer(model_dir=None):
    """v2에서 저장한 STS regression scorer를 다시 로드합니다."""
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    if model_dir is None:
        model_dir = STS_SCORER_MODEL_DIR

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(STS_SCORER_TOKENIZER_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()
    return tokenizer, model, device


def ensure_models_loaded():
    """qwen_generation_model, sts_tokenizer, sts_model, sts_device가 없으면 다시 만듭니다."""
    global qwen_generation_model
    global sts_tokenizer
    global sts_model
    global sts_device

    if "qwen_generation_model" not in globals():
        qwen_generation_model = reload_saved_qwen_adapter(QWEN_ADAPTER_DIR)

    if "sts_model" not in globals() or "sts_tokenizer" not in globals() or "sts_device" not in globals():
        sts_tokenizer, sts_model, sts_device = reload_sts_scorer(STS_SCORER_MODEL_DIR)

    return qwen_generation_model, sts_tokenizer, sts_model, sts_device


def generate_ranked_paraphrases_after_reload(sentence, num_candidates=30, top_k=10):
    """저장된 모델들을 다시 확인한 뒤 paraphrase 후보를 생성하고 STS 점수순으로 정렬합니다."""
    qwen_model_loaded, _, _, _ = ensure_models_loaded()
    ranked = generate_and_rank_paraphrases(
        sentence=sentence,
        qwen_model_for_generation=qwen_model_loaded,
        num_candidates=num_candidates,
        top_k=top_k,
    )
    return ranked




In [19]:
# 사용 예시: 아래 문장만 바꿔서 실행하면 됩니다.
input_sentence = "박대건님의 달콤 살벌한 이야기를 들으려면 제주도에 와야 한다."

ranked_results = generate_ranked_paraphrases_after_reload(
    sentence=input_sentence,
    num_candidates=30,
    top_k=10,
)

print("입력 문장:", input_sentence)
print()
print_ranked_paraphrases(ranked_results)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

입력 문장: 박대건님의 달콤 살벌한 이야기를 들으려면 제주도에 와야 한다.

01. score=4.3568 | 박대건의 달콤하고 살상적인 이야기를 듣기 위해서는 제주에 와야 합니다.
02. score=4.2868 | 박대건의 달콤하고 살상적인 이야기를 듣고 싶다면, 제주도에 와야 합니다.
03. score=4.2635 | 박대건의 달콤하고 잔혹한 이야기를 듣기 위해서는 제주에 오셔야 합니다.
04. score=4.2379 | 박대건 씨의 달콤하고 잔인한 이야기를 듣기 위해서는 제주도에 가야 합니다.
05. score=4.2284 | 박대건의 달콤하고 살상적인 이야기를 듣고 싶다면, 제주도에 오셔야 합니다.
06. score=4.2107 | 박대건씨의 달콤하고 잔혹한 이야기를 듣기 위해서는 제주도에 가야 합니다.
07. score=4.1756 | 박대건의 달콤하고 살인적인 이야기를 듣고 싶다면, 제주에 와야 합니다.
08. score=4.1694 | 박대건의 달콤하고 살인적인 이야기를 듣기 위해서는 제주도에 오셔야 합니다.
09. score=4.1497 | 박대건 님의 달콤하고 살인적인 이야기를 듣기 위해서는 제주에 오셔야 합니다.
10. score=4.1291 | 박대건의 달콤하고 살인적인 이야기를 듣기 위해서는 제주에 가야 합니다.


In [20]:
# 사용 예시: 아래 문장만 바꿔서 실행하면 됩니다.
input_sentence = "박여정님은 밤에 날라다니고 낮에 자는 올빼미 같은 사람이다."

ranked_results = generate_ranked_paraphrases_after_reload(
    sentence=input_sentence,
    num_candidates=30,
    top_k=10,
)

print("입력 문장:", input_sentence)
print()
print_ranked_paraphrases(ranked_results)

입력 문장: 박여정님은 밤에 날라다니고 낮에 자는 올빼미 같은 사람이다.

01. score=4.8392 | 박여정은 밤에는 날아다니고 낮에는 자는 올빼미 같은 사람입니다.
02. score=4.8169 | 박여정은 밤에 날아다니고 낮에 자는 올빼미 같은 사람이야.
03. score=4.6508 | 밤에는 날아다니고 낮에는 자는 올빼미와 같은 사람이 박여정입니다.
04. score=4.6132 | 박여정 씨는 밤에는 날아다니고 낮에는 잠을 자는 올빼미 같은 사람입니다.
05. score=4.5549 | 박여정은 밤에 날아다니고 낮에는 자는 올빼미 같습니다.
06. score=4.5447 | 박여정은 밤에 날아다니고 낮에 자는 올빼미 같습니다.
07. score=4.5381 | 밤에는 날아다니고 낮에는 자는 올빼미와 같은 여자입니다. 박여정은요.
08. score=4.5281 | 박여정은 밤에는 날아다니고 낮에는 자는 올빼미와 같습니다.
09. score=4.4985 | 박여정은 밤에는 날아다니고 낮에는 자는 올빼미와 같다고 합니다.
10. score=4.4925 | 박여정은 밤에는 날아다니고 낮에는 잠을 잔 올빼미와 같습니다.


In [21]:
# 사용 예시: 아래 문장만 바꿔서 실행하면 됩니다.
input_sentence = "김문주님은 제주도에서 맛있는 음식을 먹는 것을 좋아한다."

ranked_results = generate_ranked_paraphrases_after_reload(
    sentence=input_sentence,
    num_candidates=30,
    top_k=4,
)

print("입력 문장:", input_sentence)
print()
print_ranked_paraphrases(ranked_results)

입력 문장: 김문주님은 제주도에서 맛있는 음식을 먹는 것을 좋아한다.

01. score=4.9161 | 김문주 씨는 제주에서 맛있는 음식을 먹는 것을 좋아해요.
02. score=4.9134 | 김문주 씨는 제주도에서 맛있는 음식을 먹는 것을 좋아해요.
03. score=4.9051 | 김문주 씨는 제주에서 맛있는 음식을 먹는 걸 좋아해요.
04. score=4.9015 | 김문주씨는 제주에서 맛있는 음식을 먹는 것을 좋아해요.


In [22]:
# 사용 예시: 아래 문장만 바꿔서 실행하면 됩니다.
input_sentence = "Jeju Island is where you have to come to listen to Park Daegeon's sweet and bloody story."

ranked_results = generate_ranked_paraphrases_after_reload(
    sentence=input_sentence,
    num_candidates=30,
    top_k=4,
)

print("입력 문장:", input_sentence)
print()
print_ranked_paraphrases(ranked_results)

입력 문장: Jeju Island is where you have to come to listen to Park Daegeon's sweet and bloody story.

01. score=2.8188 | 제주의 이유야! Park Daegeon의 달콤하면서도 피투성이 이야기 들을ために 오셔야 하는 이유가.
02. score=2.6497 | 제주도에 오셔야 할 Park Dae Geon의 달콤한, 피어린 이야기를 들어야 합니다.
03. score=2.5344 | 제주도에서 오셔야만 Park Daegeon의 달콤하고 피로 가득한 이야기를 들을 수 있어요.
04. score=2.2666 | 제주도에서 오셔야 할 것은 Park Dae-Geon이 가진 달콤하면서도 잔혹한 이야기를 들어야 하는 것입니다.


# 8. 새 커널/다른 노트북에서 바로 쓰는 독립 실행 셀

아래 코드 셀은 앞의 셀들을 실행하지 않은 새 커널에서도 동작하도록 필요한 import, 경로 탐색, Qwen LoRA adapter 로드, STS scorer 로드, 후보 생성, 점수 계산, top 10 정렬 함수를 모두 한 번에 정의합니다. 다른 `.ipynb` 파일에서도 이 셀만 가져가서 실행할 수 있습니다.

실행 전 `trl`, `peft`, `bitsandbytes`가 설치되어 있어야 합니다. 저장된 LoRA adapter는 base Qwen 모델 전체를 포함하지 않으므로, 사용할 때마다 base model을 다시 불러오고 그 위에 adapter를 얹습니다.

In [ ]:
# 새 커널/다른 ipynb 독립 실행용 코드입니다.
# 이 셀 하나만 실행한 뒤 `generate_ranked_paraphrases_standalone(...)`을 호출하면 됩니다.

import gc
import importlib.util
import re
from pathlib import Path

import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)


def check_standalone_dependencies():
    """Qwen LoRA/QLoRA 추론에 필요한 패키지 설치 여부를 확인합니다."""
    missing_packages = [
        package_name
        for package_name in ["peft", "bitsandbytes"]
        if importlib.util.find_spec(package_name) is None
    ]
    if missing_packages:
        raise ModuleNotFoundError(
            "다음 패키지가 필요합니다: "
            + ", ".join(missing_packages)
            + "\n터미널에서 `uv pip install peft bitsandbytes`를 실행한 뒤 커널을 재시작하세요."
        )


check_standalone_dependencies()
from peft import PeftModel


QWEN_BASE_MODEL_NAME = "Qwen/Qwen3.5-4B-Base"
STS_SCORER_TOKENIZER_NAME = "klue/roberta-base"
QWEN_ADAPTER_DIR_NAME = "qwen3_5_4b_klue_sts_paraphrase_adapter"


def find_project_dir():
    """현재 작업 위치가 프로젝트 루트이든 하위 폴더이든 저장 파일이 있는 폴더를 찾습니다."""
    current = Path.cwd().resolve()
    candidates = [
        current,
        current / "huggingface_transformers_project",
        current.parent,
        current.parent / "huggingface_transformers_project",
    ]

    for candidate in candidates:
        if (candidate / QWEN_ADAPTER_DIR_NAME).exists():
            return candidate

    raise FileNotFoundError(
        "Qwen LoRA adapter 폴더를 찾지 못했습니다. `PROJECT_DIR`을 직접 지정하세요.\n"
        f"찾는 폴더명: {QWEN_ADAPTER_DIR_NAME}"
    )


PROJECT_DIR = find_project_dir()
QWEN_ADAPTER_DIR = PROJECT_DIR / QWEN_ADAPTER_DIR_NAME


def find_sts_scorer_model_dir(project_dir=PROJECT_DIR):
    """v2에서 저장된 STS regression best_model 폴더를 찾습니다."""
    candidates = list(project_dir.glob("transformers_klue_sts_regression_roberta_lr_*/best_model"))
    candidates = [path for path in candidates if (path / "config.json").exists()]

    if not candidates:
        raise FileNotFoundError(
            "STS scorer best_model 폴더를 찾지 못했습니다. `STS_SCORER_MODEL_DIR`을 직접 지정하세요."
        )

    return max(candidates, key=lambda path: path.stat().st_mtime)


STS_SCORER_MODEL_DIR = find_sts_scorer_model_dir()

print("PROJECT_DIR:", PROJECT_DIR)
print("Qwen LoRA adapter:", QWEN_ADAPTER_DIR)
print("STS scorer:", STS_SCORER_MODEL_DIR)


def build_paraphrase_prompt_standalone(sentence):
    """SFT 때 사용한 prompt 형식과 같은 형태로 입력 문장을 감쌉니다."""
    return (
        "### 지시\n"
        "다음 한국어 문장을 의미는 유지하면서 다른 표현의 한 문장으로 바꿔 쓰세요.\n"
        "출력은 paraphrase 문장 하나만 작성하세요.\n\n"
        "### 입력\n"
        f"{sentence}\n\n"
        "### 출력\n"
    )


def load_qwen_lora_for_generation(
    base_model_name=QWEN_BASE_MODEL_NAME,
    adapter_dir=QWEN_ADAPTER_DIR,
):
    """base Qwen 모델을 4-bit로 로드하고 저장된 LoRA adapter를 얹어 생성 모델을 만듭니다."""
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=compute_dtype,
        trust_remote_code=True,
    )
    base_model.config.use_cache = True
    base_model.config.pad_token_id = tokenizer.pad_token_id

    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()
    return tokenizer, model


def load_sts_scorer_standalone(
    scorer_model_dir=STS_SCORER_MODEL_DIR,
    scorer_tokenizer_name=STS_SCORER_TOKENIZER_NAME,
):
    """저장된 STS regression scorer와 tokenizer를 로드합니다."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(scorer_tokenizer_name)
    model = AutoModelForSequenceClassification.from_pretrained(scorer_model_dir).to(device)
    model.eval()
    return tokenizer, model, device


qwen_tokenizer_standalone = None
qwen_model_standalone = None
sts_tokenizer_standalone = None
sts_model_standalone = None
sts_device_standalone = None


def ensure_standalone_models_loaded():
    """필요한 모델이 메모리에 없으면 저장 경로에서 다시 불러옵니다."""
    global qwen_tokenizer_standalone
    global qwen_model_standalone
    global sts_tokenizer_standalone
    global sts_model_standalone
    global sts_device_standalone

    if qwen_model_standalone is None or qwen_tokenizer_standalone is None:
        qwen_tokenizer_standalone, qwen_model_standalone = load_qwen_lora_for_generation()

    if sts_model_standalone is None or sts_tokenizer_standalone is None:
        (
            sts_tokenizer_standalone,
            sts_model_standalone,
            sts_device_standalone,
        ) = load_sts_scorer_standalone()

    return (
        qwen_tokenizer_standalone,
        qwen_model_standalone,
        sts_tokenizer_standalone,
        sts_model_standalone,
        sts_device_standalone,
    )


def clean_paraphrase_output_standalone(text, eos_token=None):
    """생성 결과에서 prompt 조각이나 불필요한 줄을 정리합니다."""
    text = text.strip()
    if eos_token:
        text = text.replace(eos_token, "").strip()
    text = re.split(r"\n###|###|\n입력:|\n지시:", text)[0].strip()
    lines = [line.strip(" -•\t") for line in text.splitlines() if line.strip()]
    return lines[0] if lines else text


def generate_candidates_standalone(
    sentence,
    num_candidates=30,
    batch_return_sequences=5,
    max_new_tokens=80,
    temperature=0.85,
    top_p=0.92,
    repetition_penalty=1.05,
):
    """입력 문장 하나에서 paraphrase 후보를 생성합니다."""
    qwen_tokenizer, qwen_model, _, _, _ = ensure_standalone_models_loaded()
    prompt = build_paraphrase_prompt_standalone(sentence)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(qwen_model.device)

    candidates = []
    seen = set()
    attempts = 0
    max_attempts = max(10, num_candidates * 2)

    while len(candidates) < num_candidates and attempts < max_attempts:
        attempts += 1
        current_batch = min(batch_return_sequences, num_candidates - len(candidates))

        with torch.no_grad():
            outputs = qwen_model.generate(
                **inputs,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                repetition_penalty=repetition_penalty,
                max_new_tokens=max_new_tokens,
                num_return_sequences=current_batch,
                pad_token_id=qwen_tokenizer.pad_token_id,
                eos_token_id=qwen_tokenizer.eos_token_id,
            )

        input_length = inputs["input_ids"].shape[-1]
        for output in outputs:
            generated = qwen_tokenizer.decode(output[input_length:], skip_special_tokens=True)
            candidate = clean_paraphrase_output_standalone(generated, qwen_tokenizer.eos_token)
            if candidate and candidate != sentence and candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    return candidates[:num_candidates]


def score_candidates_standalone(source_sentence, candidates, batch_size=32, max_length=128):
    """기존 STS regression scorer로 후보 문장들의 유사도 점수를 계산합니다."""
    _, _, sts_tokenizer, sts_model, sts_device = ensure_standalone_models_loaded()
    scored = []

    for start in range(0, len(candidates), batch_size):
        batch_candidates = candidates[start : start + batch_size]
        encoded = sts_tokenizer(
            [source_sentence] * len(batch_candidates),
            batch_candidates,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_token_type_ids=False,
            return_tensors="pt",
        )
        encoded = {key: value.to(sts_device) for key, value in encoded.items()}

        with torch.no_grad():
            outputs = sts_model(**encoded)

        scores = outputs.logits.squeeze(-1).detach().cpu().numpy()
        scores = np.clip(scores, 0.0, 5.0)

        for candidate, score in zip(batch_candidates, scores):
            scored.append({"candidate": candidate, "score": float(score)})

    return scored


def generate_ranked_paraphrases_standalone(sentence, num_candidates=30, top_k=10):
    """후보 30개를 생성하고 STS 점수가 높은 순으로 top 10을 반환합니다."""
    candidates = generate_candidates_standalone(sentence, num_candidates=num_candidates)
    scored_candidates = score_candidates_standalone(sentence, candidates)
    return sorted(scored_candidates, key=lambda row: row["score"], reverse=True)[:top_k]


def print_ranked_paraphrases_standalone(ranked_results):
    """정렬된 paraphrase 결과를 보기 좋게 출력합니다."""
    for rank, row in enumerate(ranked_results, start=1):
        print(f"{rank:02d}. score={row['score']:.4f} | {row['candidate']}")


def clear_standalone_models():
    """독립 실행 셀에서 로드한 모델을 메모리에서 정리합니다."""
    global qwen_tokenizer_standalone
    global qwen_model_standalone
    global sts_tokenizer_standalone
    global sts_model_standalone
    global sts_device_standalone

    qwen_tokenizer_standalone = None
    qwen_model_standalone = None
    sts_tokenizer_standalone = None
    sts_model_standalone = None
    sts_device_standalone = None

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# 사용 예시:
# sentence = "박대건님의 달콤 살벌한 이야기를 들으려면 제주도에 와야 한다."
# ranked = generate_ranked_paraphrases_standalone(sentence, num_candidates=30, top_k=10)
# print_ranked_paraphrases_standalone(ranked)
print("독립 실행용 함수 준비 완료")